In [2]:
import os, json, time
import numpy as np, pandas as pd
from pathlib import Path
from openai import OpenAI
from m1_utils import *

from dotenv import load_dotenv
load_dotenv()

client = OpenAI()                       # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"                   # <- change to a model your key can use; keep fixed for C3 and C4
TEMP = 0                                # if your model rejects temperature, remove it and say so
MAX_TOKENS = 150

eval_q = pd.read_parquet(OUT / "eval_query_order.parquet")
dev = eval_q[eval_q["role"] == "dev"].reset_index(drop=True)

# Condition 3: original HyDE web-search prompt (Gao et al.)
PROSE_USER = "Please write a passage to answer the question.\nQuestion: {q}\nPassage:"

# Condition 4: structured product specification
SPEC_SYSTEM = """You convert a shopper's electronics search query into a hypothetical product listing specification, written in the vocabulary of e-commerce product listings.
Rules:
- Output ONE line of at most 60 words in the format: Product type | Brand | Key specs | Notable features
- Choose the attributes that matter for this product type (phones: RAM, storage, battery mAh, processor, display; headphones: type, connectivity, battery hours, noise cancelling; cables: connector types, length; monitors: size, resolution, refresh rate; cameras: sensor, resolution, lens; etc.).
- If the query names a brand or model, copy it exactly. Never invent a model number.
- If the query names no brand, write "Brand: any". Do not guess a brand.
- Turn vague wishes into concrete listing terms (e.g. "good battery" -> "5000mAh battery", "light" -> "lightweight").
- If the query excludes something ("without X"), write "no X" instead of listing X.
- Output only the line, no explanation."""
SPEC_SHOTS = [
    ("gaming phone good battery",
     "Smartphone | Brand: any | 6000mAh battery, 8GB RAM, 256GB storage, Snapdragon 8-series processor, 120Hz AMOLED display | gaming mode, 65W fast charging"),
    ("samsung 970 evo 1tb",
     "NVMe M.2 internal SSD | Samsung 970 EVO | 1TB capacity, PCIe 3.0 x4, up to 3500MB/s read | solid state drive for laptop and desktop"),
    ("wireless earbuds for running",
     "True wireless earbuds | Brand: any | Bluetooth 5.0, IPX7 waterproof, 8 hours battery with charging case | sweatproof sports earphones with secure fit"),
]

def msgs(kind, q):
    if kind == "prose":
        return [{"role": "user", "content": PROSE_USER.format(q=q)}]
    m = [{"role": "system", "content": SPEC_SYSTEM}]
    for sq, sa in SPEC_SHOTS:
        m += [{"role": "user", "content": sq}, {"role": "assistant", "content": sa}]
    return m + [{"role": "user", "content": q}]

def chat(messages):
    last = None
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages, temperature=TEMP, max_tokens=MAX_TOKENS)
            return r.choices[0].message.content.strip()
        except Exception as e:
            last = e; time.sleep(2 ** attempt)
    raise RuntimeError(f"API failed: {last}")

def load_cache(path):
    d = {}
    if path.exists():
        for line in path.open(encoding="utf-8"):
            r = json.loads(line); d[r["query_id"]] = r["text"]
    return d

samp = dev.sample(6, random_state=3)
for qid, qtext, src in zip(samp["query_id"], samp["query"], samp["source"]):
    print(f"\nQUERY ({src}): {qtext}")
    print("  PROSE:", chat(msgs("prose", qtext)).replace("\n", " ")[:300])
    print("  SPEC :", chat(msgs("spec", qtext)))


QUERY (other): curve led monitor
  PROSE: A curved LED monitor is a type of display that features a concave shape, designed to enhance the viewing experience by providing a more immersive and engaging visual environment. Unlike traditional flat monitors, curved screens wrap around the viewer's field of vision, which can help reduce distorti
  SPEC : Curved LED monitor | Brand: any | 27-inch size, 2560x1440 resolution, 144Hz refresh rate | immersive viewing experience with ultra-thin bezels

QUERY (other): lg tone triumph
  PROSE: The term "lg tone triumph" can be interpreted in various ways, depending on the context in which it is used. In the realm of technology, particularly in audio and sound systems, "lg tone" refers to the audio products developed by LG Electronics, known for their innovative features and high-quality s
  SPEC : Bluetooth headphones | LG Tone Triumph | In-ear design, Bluetooth 4.1, 15 hours battery life | neckband style, voice assistant support, magnetic earbuds


In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import torch

PV = "v1"
def generate(kind, rows_df, workers=8):
    path = OUT / f"hyde_dev_{kind}_{PV}.jsonl"
    cache = load_cache(path)
    todo = [(qid, q) for qid, q in zip(rows_df["query_id"], rows_df["query"]) if qid not in cache]
    print(kind, "cached", len(cache), "| todo", len(todo))
    with ThreadPoolExecutor(workers) as ex, path.open("a", encoding="utf-8") as f:
        futs = {ex.submit(chat, msgs(kind, q)): qid for qid, q in todo}
        for fu in as_completed(futs):
            qid = futs[fu]; txt = fu.result(); cache[qid] = txt
            f.write(json.dumps({"query_id": int(qid), "text": txt}) + "\n"); f.flush()
    return cache

t0 = time.time()
prose = generate("prose", dev); spec = generate("spec", dev)
print(f"generation done {time.time()-t0:.0f}s")

# ---- embed generated text with the SAME model/index used for C2
queries, judg, corpus = load_all(); qrels = build_qrels(judg); pids = corpus["product_id"].to_numpy()
doc_emb = np.load(OUT / "doc_emb_minilm.npy")
bm25_scores = np.load(OUT / "bm25_scores.npy", mmap_mode="r")
dense_raw = np.load(OUT / "dense_raw_scores.npy", mmap_mode="r")
dev_rows = eval_q.index[eval_q["role"] == "dev"]
qids = eval_q.loc[dev_rows, "query_id"].tolist()

dev_device = "cuda" if torch.cuda.is_available() else "cpu"
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev_device); st.max_seq_length = 256
def dense_from(texts):
    return st.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32") @ doc_emb.T
D = {"C2 raw query": np.asarray(dense_raw)[dev_rows.to_numpy()],
     "C3 prose HyDE": dense_from([prose[q] for q in qids]),
     "C4 spec HyDE": dense_from([spec[q] for q in qids])}
BM = np.asarray(bm25_scores)[dev_rows.to_numpy()]          # rows aligned with qids

def top10(scores_row): 
    t = np.argpartition(-scores_row, 10)[:10]; return list(pids[t[np.argsort(-scores_row[t])]])
def fuse_row(b, d, a, K=100):
    cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
    s = a * minmax(b[cand]) + (1 - a) * minmax(d[cand])
    return list(pids[cand[np.argsort(-s)[:10]]])

def ev_dense(M):      return evaluate({q: top10(M[i]) for i, q in enumerate(qids)}, qrels)["ndcg@10"]
def ev_fused(M, a):   return evaluate({q: fuse_row(BM[i], M[i], a) for i, q in enumerate(qids)}, qrels)["ndcg@10"]

bm = evaluate({q: top10(BM[i]) for i, q in enumerate(qids)}, qrels)
rows = {"C1 BM25 only": bm["ndcg@10"]}
for k in D: rows[k + " (dense only)"] = ev_dense(D[k])
print("DENSE-ONLY retrieval, dev NDCG@10 / MRR@10:")
tab = pd.DataFrame({k: v for k, v in rows.items()}).mean().round(4); print(tab.to_string())

alphas = np.round(np.linspace(0, 1, 11), 1)
grid = {k: [ev_fused(D[k], a).mean() for a in alphas] for k in D}
print("\nFUSED with BM25 (fixed alpha), mean NDCG@10:")
print(pd.DataFrame(grid, index=alphas).round(4).to_string())
best = {k: alphas[int(np.argmax(v))] for k, v in grid.items()}; print("best fixed alpha per dense variant:", best)

# ---- paired bootstrap: spec vs prose (dense-only and fused at each variant's own best alpha)
def boot(a, b, n=5000, seed=0):
    d = (a - b).values; r = np.random.default_rng(seed)
    m = np.array([d[r.integers(0, len(d), len(d))].mean() for _ in range(n)])
    return d.mean(), np.percentile(m, [2.5, 97.5])
for name, (x, y) in {"dense-only spec-prose": (ev_dense(D["C4 spec HyDE"]), ev_dense(D["C3 prose HyDE"])),
                     "fused(best a) spec-prose": (ev_fused(D["C4 spec HyDE"], best["C4 spec HyDE"]), ev_fused(D["C3 prose HyDE"], best["C3 prose HyDE"]))}.items():
    m, ci = boot(x, y); print(f"{name}: diff {m:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")

# ---- ROUGE-L of generated text vs Exact-match product docs (up to 5 per query)
doc_of = dict(zip(corpus["product_id"], corpus["doc_text"]))
exact = judg[judg["esci_label"] == "E"].groupby("query_id")["product_id"].apply(list)
sc = rouge_scorer.RougeScorer(["rougeL"])
def rl(gen):
    f, r = [], []
    for q in qids:
        s = [sc.score(doc_of[p], gen[q]) ["rougeL"] for p in exact[q][:5]]
        f.append(np.mean([x.fmeasure for x in s])); r.append(np.mean([x.recall for x in s]))
    return np.mean(f), np.mean(r)
print("\nROUGE-L vs Exact docs (F1 / recall):  prose %.3f / %.3f   spec %.3f / %.3f" % (*rl(prose), *rl(spec)))
print("avg words: prose %.0f | spec %.0f" % (np.mean([len(prose[q].split()) for q in qids]), np.mean([len(spec[q].split()) for q in qids])))

# ---- by query type: negation and model-token queries
src = eval_q.loc[dev_rows, "source"].values
neg = np.array(["without" in t.lower() for t in eval_q.loc[dev_rows, "query"]])
for name, m in [("queries with 'without'", neg), ("others", ~neg)]:
    print(f"{name} (n={m.sum()}): prose {ev_dense(D['C3 prose HyDE'])[m].mean():.3f} | spec {ev_dense(D['C4 spec HyDE'])[m].mean():.3f} | raw {ev_dense(D['C2 raw query'])[m].mean():.3f} | BM25 {bm['ndcg@10'][m].mean():.3f}")

# ---- 50 specs for the manual hallucination review promised in the proposal
rev = pd.DataFrame({"query_id": qids, "query": eval_q.loc[dev_rows, "query"].values, "spec": [spec[q] for q in qids],
                    "an_exact_product": [doc_of[exact[q][0]][:160] for q in qids]}).sample(50, random_state=1)
rev.assign(spec_ok="", note="").to_csv(OUT / "spec_review_50.csv", index=False); print("saved spec_review_50.csv")

prose cached 0 | todo 500
spec cached 0 | todo 500
generation done 218s


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

DENSE-ONLY retrieval, dev NDCG@10 / MRR@10:
C1 BM25 only                  0.2503
C2 raw query (dense only)     0.2347
C3 prose HyDE (dense only)    0.2358
C4 spec HyDE (dense only)     0.1681

FUSED with BM25 (fixed alpha), mean NDCG@10:
     C2 raw query  C3 prose HyDE  C4 spec HyDE
0.0        0.2347         0.2358        0.1681
0.1        0.2597         0.2638        0.2146
0.2        0.2771         0.2873        0.2572
0.3        0.2911         0.3026        0.2825
0.4        0.2950         0.3123        0.2948
0.5        0.2911         0.3107        0.2965
0.6        0.2864         0.3005        0.2938
0.7        0.2751         0.2886        0.2876
0.8        0.2698         0.2779        0.2739
0.9        0.2599         0.2634        0.2648
1.0        0.2501         0.2501        0.2505
best fixed alpha per dense variant: {'C2 raw query': np.float64(0.4), 'C3 prose HyDE': np.float64(0.4), 'C4 spec HyDE': np.float64(0.5)}
dense-only spec-prose: diff -0.0677  95% CI [-0.0880, -0.0468

In [4]:
# --- 1) does the spec drop the query's own words?
qtext_dev = eval_q.loc[dev_rows, "query"].tolist()
qtok = [set(tokenize(t)) for t in qtext_dev]
ret = lambda gen: np.array([len(qt & set(tokenize(gen[q]))) / max(len(qt), 1) for qt, q in zip(qtok, qids)])
r_prose, r_spec = ret(prose), ret(spec)
print(f"share of query tokens kept in generated text: prose {r_prose.mean():.2f} | spec {r_spec.mean():.2f}")

d_prose, d_spec = ev_dense(D["C3 prose HyDE"]).values, ev_dense(D["C4 spec HyDE"]).values
has_model = np.array([any(re.search(r"[a-z]", w) and re.search(r"\d", w) for w in re.findall(r"[a-z0-9-]+", t.lower())) for t in qtext_dev])
nw = np.array([len(t.split()) for t in qtext_dev])
grp = pd.DataFrame({"prose": d_prose, "spec": d_spec, "retain_spec": pd.cut(r_spec, [-.01, .5, .99, 1.0], labels=["<50%", "50-99%", "100%"]),
                    "model_token": has_model, "len": pd.cut(nw, [0, 2, 4, 100], labels=["1-2w", "3-4w", "5+w"])})
for c in ["retain_spec", "model_token", "len"]:
    print(f"\ndense-only NDCG@10 by {c}:"); print(grp.groupby(c, observed=True)[["prose", "spec"]].agg(["mean"]).round(3).assign(n=grp[c].value_counts()))

# --- 2) variants (scores are linear in the embeddings, so averaging scores = averaging embeddings)
raw, pro, spc = D["C2 raw query"], D["C3 prose HyDE"], D["C4 spec HyDE"]
Dv = {"C3 prose": pro, "C4 spec": spc,
      "C3 + query (avg emb)": (raw + pro) / 2, "C4 + query (avg emb)": (raw + spc) / 2,
      "C3 query+prose (concat)": dense_from([f"{q} | {prose[i]}" for q, i in zip(qtext_dev, qids)]),
      "C4 query+spec (concat)": dense_from([f"{q} | {spec[i]}" for q, i in zip(qtext_dev, qids)])}

dense_only = {k: ev_dense(v).mean() for k, v in Dv.items()}
fused = {k: [ev_fused(v, a).mean() for a in alphas] for k, v in Dv.items()}
out = pd.DataFrame({"dense_only": dense_only,
                    "best_fixed_alpha": {k: alphas[int(np.argmax(v))] for k, v in fused.items()},
                    "fused_best": {k: max(v) for k, v in fused.items()}}).round(4)
print("\nVARIANTS (dev NDCG@10) - reference: raw-query fused best = 0.2950\n"); print(out.to_string())

best_a = {k: alphas[int(np.argmax(v))] for k, v in fused.items()}
for a_, b_ in [("C4 + query (avg emb)", "C3 + query (avg emb)"), ("C4 query+spec (concat)", "C3 query+prose (concat)")]:
    for tag, x, y in [("dense-only", ev_dense(Dv[a_]), ev_dense(Dv[b_])),
                      ("fused best-a", ev_fused(Dv[a_], best_a[a_]), ev_fused(Dv[b_], best_a[b_]))]:
        m, ci = boot(x, y); print(f"{a_} - {b_} [{tag}]: {m:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")

share of query tokens kept in generated text: prose 0.93 | spec 0.77

dense-only NDCG@10 by retain_spec:
             prose   spec    n
              mean   mean     
retain_spec                   
<50%         0.167  0.114  124
50-99%       0.197  0.131  116
100%         0.286  0.211  260

dense-only NDCG@10 by model_token:
             prose   spec    n
              mean   mean     
model_token                   
False        0.221  0.162  424
True         0.319  0.201   76

dense-only NDCG@10 by len:
      prose   spec    n
       mean   mean     
len                    
1-2w  0.218  0.158  141
3-4w  0.244  0.178  267
5+w   0.240  0.156   92

VARIANTS (dev NDCG@10) - reference: raw-query fused best = 0.2950

                         dense_only  best_fixed_alpha  fused_best
C3 prose                     0.2358               0.4      0.3123
C4 spec                      0.1681               0.5      0.2965
C3 + query (avg emb)         0.2645               0.4      0.3168
C4 + query (av

In [5]:
np.save(OUT / "dense_dev_C3avg_v1.npy", Dv["C3 + query (avg emb)"].astype("float32"))
np.save(OUT / "dense_dev_C4avg_v1.npy", Dv["C4 + query (avg emb)"].astype("float32"))
np.save(OUT / "dense_dev_qids.npy", np.array(qids))

base = ev_fused(raw, 0.4)
for k in ["C3 + query (avg emb)", "C4 + query (avg emb)"]:
    m, ci = boot(ev_fused(Dv[k], 0.4), base)
    print(f"{k} vs raw-query dense, both fused at alpha=0.4: {m:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")

C3 + query (avg emb) vs raw-query dense, both fused at alpha=0.4: +0.0218  95% CI [+0.0137, +0.0306]
C4 + query (avg emb) vs raw-query dense, both fused at alpha=0.4: +0.0218  95% CI [+0.0122, +0.0317]
